In [7]:
# 1. Install the Gurobi library
%pip install gurobipy

import gurobipy as gp
from gurobipy import *

# 2. Input your license credentials
params = {
    "WLSACCESSID": 'a9386248-233e-42c3-9f08-56b178e9f9e2',
    "WLSSECRET": 'a94f121b-7d29-44d4-ba07-aeea9c9954f1',
    "LICENSEID": 2769033,
}

# 3. Create the Gurobi environment
env = gp.Env(params=params)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2769033
Academic license 2769033 - for non-commercial use only - registered to pa___@smu.ca


In [11]:
!pip -q install pulp openpyxl

import pandas as pd
import numpy as np
import pulp
from google.colab import files


excel_file = "/content/Params (1).xlsx"  # change if it shows "Params (1).xlsx"

A = pd.read_excel(excel_file, sheet_name="A-Matrix", header=None).to_numpy(dtype=float)
b = pd.read_excel(excel_file, sheet_name="b-Vector", header=None).to_numpy(dtype=float).flatten()
c = pd.read_excel(excel_file, sheet_name="c-Vector", header=None).to_numpy(dtype=float).flatten()

print(A.shape, len(b), len(c))


(20, 10) 20 10


Define Min_Vector() to build the primal minimization model, name variables
𝑥
x, add constraints and objective, and write the model to Primal.lp.

In [12]:
def Min_Vector(A, b, c):
    # Create a minimization LP model
    model = pulp.LpProblem("Primal", pulp.LpMinimize)

    m, n = A.shape  # m constraints, n variables

    # Decision variables: x0, x1, ..., x(n-1), with x >= 0
    x = []
    for j in range(n):
        x.append(pulp.LpVariable(f"x{j}", lowBound=0))

    # Objective: minimize sum(c[j] * x[j])
    model += pulp.lpSum(c[j] * x[j] for j in range(n))

    # Constraints: A x >= b
    for i in range(m):
        model += pulp.lpSum(A[i, j] * x[j] for j in range(n)) >= b[i]

    # Export LP file
    model.writeLP("Primal.lp")

    return model


Solve the primal model and display solver status.

In [13]:
primal_model = Min_Vector(A, b, c)
status = primal_model.solve(pulp.PULP_CBC_CMD(msg=False))
print("Primal status:", pulp.LpStatus[status])


Primal status: Optimal


Define print_solution() to print only positive decision variables and the objective value.

In [14]:
def print_solution(model, tol=1e-8):
    print("Positive decision variables:")
    for v in model.variables():
        if v.value() is not None and v.value() > tol:
            print(v.name, "=", v.value())
    print("Objective value =", pulp.value(model.objective))


In [15]:
print("=== PRIMAL SOLUTION ===")
print_solution(primal_model)
primal_obj = pulp.value(primal_model.objective)


=== PRIMAL SOLUTION ===
Positive decision variables:
x3 = 66.614907
x5 = 32.291925
x8 = 24.018634
Objective value = 5379.335404


In [16]:
files.download("Primal.lp")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>